In [78]:
import pandas as pd
# calucalte MMC and confusion matrix libraries
from sklearn.metrics import matthews_corrcoef, confusion_matrix, recall_score, precision_score




In [ ]:
rnaSeq_origial =  pd.read_csv('Data/RNAseq_abundances_adjusted_combat_inmose.csv')
seq_metadata_path = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Data/All_rna_samples_metadata_edit.csv"
seq_metadata = pd.read_csv(seq_metadata_path)
rnaSeq_origial= pd.merge(rnaSeq_origial, seq_metadata, on='Sample', how='inner')
females = rnaSeq_origial[rnaSeq_origial['Sex_x'] == 'female']
young_females = females[females['Age'] < 35]
old_females = females[females['Age'] > 65]
middle_aged_females = females[(females['Age'] >= 25) & (females['Age'] <= 65)]

Taken from /home/karen/Documents/phd/Files for theses/Papers/Machine learning for aged muscle/Results/models_results/Eval_b'


Catboost matrix

Data distribution age_ranges Test data
[{'label': '18-26', 'range': (18, 26), 'count': 18}, early young
 {'label': '26-35', 'range': (26, 35), 'count': 19}, young
 {'label': '35-65', 'range': (35, 65), 'count': 18}, middle age
 {'label': '65-71', 'range': (65, 71), 'count': 6}, early old
 {'label': '71-72', 'range': (71, 72), 'count': 5}, old
 {'label': '72-95', 'range': (72, 95), 'count': 26}] late old


 array([[16,  2,  0,  0,  0,  0],
       [ 4, 11,  4,  0,  0,  0],
       [ 0,  0, 18,  0,  0,  0],
       [ 0,  0,  1,  4,  1,  0],
       [ 0,  0,  0,  5,  0,  0],
       [ 0,  0,  5,  8,  0, 13]])

In [58]:
catboost_results = '/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/master/Catboost/catboost_smote_7b_untoached_set_results.csv'
ridge_results = '//home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/master/Ridge/Ridge_90-10_results_validation_untoached_Ridge.csv'

catboost = pd.read_csv(catboost_results, index_col=0)
ridge = pd.read_csv(ridge_results)



In [59]:
# divide the Actual and Predicted columns (numerical) into the early-young, young, middle-age, early-old, old, late-old categories
#  (18, 26), (26, 35), (35, 65), (65, 71), (71, 72), (72, 95)
def categorize_age(age):
    if 18 <= age < 26:
        return 'early-young'
    elif 26 <= age < 35:
        return 'young'
    elif 35 <= age <= 65:
        return 'middle-age'
    elif 65 < age <= 71:
        return 'early-old'
    elif 71 < age <= 72:
        return 'old'
    elif 72 < age <= 95:
        return 'late-old'
    else:
        return 'unknown'
catboost['Actual_Category'] = catboost['Actual'].apply(categorize_age)
catboost['Predicted_Category'] = catboost['Predicted'].apply(categorize_age)
ridge['Actual_Category'] = ridge['Actual'].apply(categorize_age)
ridge['Predicted_Category'] = ridge['Predicted'].apply(categorize_age)



In [60]:
catboost.columns

Index(['Actual', 'Predicted', 'Experiment', 'Actual_Category',
       'Predicted_Category'],
      dtype='object')

In [61]:
ridge.columns

Index(['Unnamed: 0', 'Actual', 'Status', 'Experiment', 'Predicted',
       'Actual_Category', 'Predicted_Category'],
      dtype='object')

In [62]:
# get confusion matrix (actual vs predicted)
catboost_cm = confusion_matrix(catboost['Actual_Category'], catboost['Predicted_Category'])
ridge_cm = confusion_matrix(ridge['Actual_Category'], ridge['Predicted_Category'])

catboost_cm

array([[20,  0,  8,  4,  1,  0],
       [ 0, 11,  0,  0,  0,  2],
       [ 4,  0, 10,  1,  2,  0],
       [ 0,  0,  0, 18,  0,  0],
       [ 1,  0,  3,  0,  0,  0],
       [ 0,  1,  0,  3,  0, 19]])

In [66]:
ridge_cm

array([[15,  0, 13,  3,  2,  0,  0],
       [ 0, 10,  0,  0,  0,  1,  2],
       [ 2,  0, 10,  3,  1,  1,  0],
       [ 0,  0,  2, 16,  0,  0,  0],
       [ 0,  0,  2,  2,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0],
       [ 0,  5,  0,  4,  0,  2, 12]])

In [63]:
catboost_mmc = matthews_corrcoef(catboost['Actual_Category'], catboost['Predicted_Category'])
ridge_mmc = matthews_corrcoef(ridge['Actual_Category'], ridge['Predicted_Category'])

catboost_mmc, ridge_mmc

(0.6592438396509925, 0.5091290701011952)

In [80]:
catboost_precision = precision_score(catboost['Actual_Category'], catboost['Predicted_Category'], average=None, zero_division=0)
ridge_precision = precision_score(ridge['Actual_Category'], ridge['Predicted_Category'], average=None, zero_division=0)
catboost_recall = recall_score(catboost['Actual_Category'], catboost['Predicted_Category'], average=None, zero_division=0)
ridge_recall = recall_score(ridge['Actual_Category'], ridge['Predicted_Category'], average=None, zero_division=0)

print("CatBoost Precision per class:", catboost_precision)

print("CatBoost Recall per class:", catboost_recall)
print("\n")
print("Ridge Precision per class:", ridge_precision)
print("Ridge Recall per class:", ridge_recall)


CatBoost Precision per class: [0.8        0.91666667 0.47619048 0.69230769 0.         0.9047619 ]
CatBoost Recall per class: [0.60606061 0.84615385 0.58823529 1.         0.         0.82608696]


Ridge Precision per class: [0.88235294 0.66666667 0.37037037 0.57142857 0.         0.
 0.85714286]
Ridge Recall per class: [0.45454545 0.76923077 0.58823529 0.88888889 0.         0.
 0.52173913]


In [ ]:
# average sensitivity and specificity for both models
ridge_sensitivity = []
ridge_specificity = []
catboost_sensitivity = []
catboost_specificity = []
for i in range(len(catboost_cm)):
    # Sensitivity, hit rate, recall, or true positive rate
    catboost_sens = catboost_cm[i, i] / sum(catboost_cm[i, :]) if sum(catboost_cm[i, :]) > 0 else 0
    ridge_sens = ridge_cm[i, i] / sum(ridge_cm[i, :]) if sum(ridge_cm[i, :]) > 0 else 0
    catboost_sensitivity.append(catboost_sens)
    ridge_sensitivity.append(ridge_sens)
    
    # Specificity or true negative rate
    catboost_spec = sum(catboost_cm[j, j] for j in range(len(catboost_cm)) if j != i) / sum(catboost_cm[:, j].sum() for j in range(len(catboost_cm)) if j != i) if sum(catboost_cm[:, j].sum() for j in range(len(catboost_cm)) if j != i) > 0 else 0
    ridge_spec = sum(ridge_cm[j, j] for j in range(len(ridge_cm)) if j != i) / sum(ridge_cm[:, j].sum() for j in range(len(ridge_cm)) if j != i) if sum(ridge_cm[:, j].sum() for j in range(len(ridge_cm)) if j != i) > 0 else 0
    catboost_specificity.append(catboost_spec)
    ridge_specificity.append(ridge_spec)

In [64]:
# calculate the r2 and mean absolute error for both models
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
catboost_r2 = r2_score(catboost['Actual'], catboost['Predicted'])
catboost_mae = mean_absolute_error(catboost['Actual'], catboost['Predicted'])
catboost_rmse = mean_squared_error(catboost['Actual'], catboost['Predicted'], squared=False)
ridge_r2 = r2_score(ridge['Actual'], ridge['Predicted'])
ridge_mae = mean_absolute_error(ridge['Actual'], ridge['Predicted'])
ridge_rmse = mean_squared_error(ridge['Actual'], ridge['Predicted'], squared=False)


print(f"catboost - R2: {catboost_r2}, MAE: {catboost_mae}, RMSE: {catboost_rmse}")
print(f"ridge - R2: {ridge_r2}, MAE: {ridge_mae}, RMSE: {ridge_rmse}")


catboost - R2: 0.9063860346006213, MAE: 5.05377536006394, RMSE: 6.846056123233672
ridge - R2: 0.8532649273316048, MAE: 6.3676759502682945, RMSE: 8.571110613604624


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [65]:
ridge_r2

0.8532649273316048

In [70]:
def get_sensitivity_specificity(cm):
    sensitivity = []
    specificity = []
    for i in range(len(cm)):
        tp = cm[i, i]
        fn = sum(cm[i, :]) - tp
        fp = sum(cm[:, i]) - tp
        tn = cm.sum() - (tp + fn + fp)
        sensitivity.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    return sensitivity, specificity

In [75]:
# get the sensitivity and specificity for both models in each age category
print("Ridge Model Sensitivity and Specificity:")
ridge_sensitivity, ridge_specificity = get_sensitivity_specificity(ridge_cm)   
for age_group in ['early-young', 'young', 'middle-age', 'early-old', 'old', 'late-old']:
    idx = ['early-young', 'young', 'middle-age', 'early-old', 'old', 'late-old'].index(age_group)
    
    print(f"{age_group} - Ridge Sensitivity: {ridge_sensitivity[idx]}, Specificity: {ridge_specificity[idx]}")

print("\n")
print("Catboost Model Sensitivity and Specificity:")
catboost_sensitivity, catboost_specificity = get_sensitivity_specificity(catboost_cm)
for age_group in ['early-young', 'young', 'middle-age', 'early-old', 'old', 'late-old']:
    idx = ['early-young', 'young', 'middle-age', 'early-old', 'old', 'late-old'].index(age_group)
    
    print(f"{age_group} - CatBoost Sensitivity: {catboost_sensitivity[idx]}, Specificity: {catboost_specificity[idx]}")
print("\n")



Ridge Model Sensitivity and Specificity:
early-young - Ridge Sensitivity: 0.45454545454545453, Specificity: 0.9733333333333334
young - Ridge Sensitivity: 0.7692307692307693, Specificity: 0.9473684210526315
middle-age - Ridge Sensitivity: 0.5882352941176471, Specificity: 0.8131868131868132
early-old - Ridge Sensitivity: 0.8888888888888888, Specificity: 0.8666666666666667
old - Ridge Sensitivity: 0.0, Specificity: 0.9711538461538461
late-old - Ridge Sensitivity: 0, Specificity: 0.9629629629629629


Catboost Model Sensitivity and Specificity:
early-young - CatBoost Sensitivity: 0.6060606060606061, Specificity: 0.9333333333333333
young - CatBoost Sensitivity: 0.8461538461538461, Specificity: 0.9894736842105263
middle-age - CatBoost Sensitivity: 0.5882352941176471, Specificity: 0.8791208791208791
early-old - CatBoost Sensitivity: 1.0, Specificity: 0.9111111111111111
old - CatBoost Sensitivity: 0.0, Specificity: 0.9711538461538461
late-old - CatBoost Sensitivity: 0.8260869565217391, Specific

In [76]:
# average sensitivity and specificity for both models
ridge_avg_sensitivity = sum(ridge_sensitivity) / len(ridge_sensitivity)
ridge_avg_specificity = sum(ridge_specificity) / len(ridge_specificity)
catboost_avg_sensitivity = sum(catboost_sensitivity) / len(catboost_sensitivity)
catboost_avg_specificity = sum(catboost_specificity) / len(catboost_specificity)
print(f"Ridge Average Sensitivity: {ridge_avg_sensitivity}, Average Specificity: {ridge_avg_specificity}")
print(f"CatBoost Average Sensitivity: {catboost_avg_sensitivity}, Average Specificity: {catboost_avg_specificity}")

Ridge Average Sensitivity: 0.4603770767453632, Average Specificity: 0.9301632330845069
CatBoost Average Sensitivity: 0.6444227838089731, Average Specificity: 0.9434439070274984


In [77]:
# sum diagonal values of confusion matrix
ridge_sum_diagonal = sum(ridge_cm[i, i] for i in range(len(ridge_cm)))
catboost_sum_diagonal = sum(catboost_cm[i, i] for i in range(len(catboost_cm)))
print(f"Ridge Sum of Diagonal: {ridge_sum_diagonal}")
print(f"CatBoost Sum of Diagonal: {catboost_sum_diagonal}")

Ridge Sum of Diagonal: 63
CatBoost Sum of Diagonal: 78
